In [1]:
import pandas as pd
import numpy as np
import re 

In [ ]:
# import urllib.request
# url = """https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"""
# file_path = "the-verdict.txt"
# urllib.request.urlretrieve(url, file_path)

In [2]:
with open("the-verdict.txt", "r", encoding='utf-8') as f:
    raw_text = f.read()

In [3]:
raw_text[:500]

'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)\n\n"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it\''

# Tokenization Based on Byte-Pair_Encoding

In [4]:
import tiktoken
tokenize = tiktoken.get_encoding("gpt2")


In [5]:
print("Vocab Size:", tokenize.n_vocab)

Vocab Size: 50257


In [6]:
ids = tokenize.encode("I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow")
ids

[40,
 367,
 2885,
 1464,
 1807,
 3619,
 402,
 271,
 10899,
 2138,
 257,
 7026,
 15632,
 438,
 2016,
 257,
 922,
 5891]

In [7]:
text_back = tokenize.decode(ids)
text_back

'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow'

# Dataset Build

In [8]:
import torch
from torch.utils.data import Dataset, DataLoader


class DatasetSpaceToken(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        
        token_ids = tokenizer.encode(text)
        
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i+max_length]
            self.input_ids.append(torch.tensor(input_chunk))
            
            target_chunk = token_ids[i+1 : i+1+max_length]
            self.target_ids.append(torch.tensor(target_chunk))
    
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]
            

In [9]:
def create_dataloader(text, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = DatasetSpaceToken(text, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dataloader

In [10]:
dataloader = create_dataloader(
    raw_text,
    batch_size=8,
    max_length=4,
    stride=4,
    shuffle=False
)

In [11]:
data_iter = iter(dataloader)
next(data_iter)

[tensor([[   40,   367,  2885,  1464],
         [ 1807,  3619,   402,   271],
         [10899,  2138,   257,  7026],
         [15632,   438,  2016,   257],
         [  922,  5891,  1576,   438],
         [  568,   340,   373,   645],
         [ 1049,  5975,   284,   502],
         [  284,  3285,   326,    11]]),
 tensor([[  367,  2885,  1464,  1807],
         [ 3619,   402,   271, 10899],
         [ 2138,   257,  7026, 15632],
         [  438,  2016,   257,   922],
         [ 5891,  1576,   438,   568],
         [  340,   373,   645,  1049],
         [ 5975,   284,   502,   284],
         [ 3285,   326,    11,   287]])]

# Token Embedding

In [12]:
vocab_size = tokenize.n_vocab
output_dim = 256
print(vocab_size, output_dim)

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

50257 256


In [13]:
input_data_first_batch = None
target_data_first_batch = None

for input_data, output_fetch in dataloader:
    input_data_first_batch, target_data_first_batch = input_data, output_fetch
    break

In [14]:
token_embedding_first_batch = token_embedding_layer(input_data_first_batch)
token_embedding_first_batch

tensor([[[-0.8279, -0.3642,  0.3995,  ..., -0.1247,  1.4323,  0.4795],
         [ 0.6074, -0.9051,  0.6571,  ...,  0.0226, -0.4034,  0.5145],
         [-0.2396, -0.3641, -0.6145,  ...,  0.8762, -0.1254,  1.2289],
         [ 1.2481,  0.0250,  0.7267,  ...,  0.1180, -0.0240,  0.3270]],

        [[ 1.1582,  0.2650,  0.7630,  ...,  0.5816,  0.2842,  0.8999],
         [-0.3320,  1.1587,  0.3495,  ...,  0.1193, -0.8565,  1.6162],
         [-0.0048,  2.0862, -1.6797,  ..., -0.3386, -1.3531, -0.4097],
         [-0.1368,  1.1638, -1.4794,  ..., -0.0691, -0.8212, -0.6275]],

        [[ 2.0077, -1.0551,  0.2618,  ..., -1.7929,  0.6990,  1.5159],
         [-0.4666,  0.1114,  0.3809,  ...,  0.0710, -1.3177,  1.1845],
         [-1.2598, -0.2760, -0.1842,  ...,  0.6693, -1.1956,  1.9330],
         [-0.7088,  0.5897,  1.2803,  ...,  0.9660, -0.8105, -0.2177]],

        ...,

        [[-1.2494,  0.0911, -0.4280,  ..., -0.2199,  0.4185,  1.6470],
         [-0.7829, -0.0945, -0.1739,  ...,  1.3690,  1.67

In [15]:
print(input_data_first_batch.shape)
print(token_embedding_first_batch.shape)

torch.Size([8, 4])
torch.Size([8, 4, 256])
